#PAEdu - Plataforma de Accesibilidad Educativa

###Instalación de librerías necesarias.
En primer lugar se instalan las dependencias requeridas para la ejecución de la aplicación.

In [1]:
# Instalamos todo lo que necesito para el modelo de lenguaje y la interfaz (Gradio)
!pip -q install transformers torch accelerate gradio sentencepiece
# PyMuPDF (fitz) para poder leer y extraer texto de los PDFs que suba el usuario
!pip -q install pymupdf
# reportlab lo uso luego para generar el PDF de salida con el texto adaptado
!pip -q install reportlab
# kokoro + soundfile son para la parte de texto a voz (TTS)
!pip install -q kokoro soundfile
# misaki hace falta como dependencia de kokoro para el inglés
!pip install -q misaki[en]
# wordfreq la uso para calcular la frecuencia de las palabras
!pip install wordfreq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 45.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.3/48.3 kB 5.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 88.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.7/82.7 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 237.9/237.9 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 734.0/734.0 kB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.7/216.7 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━

###Importación de dependencias
Carga de las librerías requeridas para el modelo y la interfaz de la aplicación.

In [2]:
import os
from datetime import datetime

import torch
import gradio as gr
import fitz  # PyMuPDF, para leer el contenido de los PDFs
import spacy
from wordfreq import zipf_frequency

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from kokoro import KPipeline
import soundfile as sf

from reportlab.lib.pagesizes import letter
from reportlab.lib.units import cm
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_CENTER, TA_JUSTIFY
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Image, HRFlowable, Table, TableStyle
)
from xml.sax.saxutils import escape

from google.colab import drive

### Clase: PDFReader
Se encarga únicamente de leer el texto de un PDF subido.

In [3]:
class PDFReader:
    """Lee el contenido de texto de un archivo PDF."""

    @staticmethod
    def leer_pdf(file) -> str:
        if file is None:
            return ""

        doc = fitz.open(file.name)
        texto = ""

        for page in doc:
            texto += page.get_text()

        doc.close()
        return texto.strip()

###Clase: TextSimplifier
Encapsula el modelo de lenguaje (flan-t5-base) y todas las tareas de reescritura de texto: simplificar, adaptar preguntas de examen y lectura fácil.

In [4]:
class TextSimplifier:
    """Carga el modelo de lenguaje y ofrece los distintos modos de adaptación de texto."""

    def __init__(self, model_name: str = "google/flan-t5-base"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = self.model.to(self.device)

        print(f"Modelo cargado en: {self.device}")

    def _generar_respuesta(self, prompt: str) -> str:
        """Método interno que manda un prompt al modelo y devuelve el texto generado."""
        inputs = self.tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=1024
        )

        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        outputs = self.model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=True,
            temperature=0.3
        )

        return self.tokenizer.decode(outputs[0], skip_special_tokens=True)

    def simplificar(self, texto: str) -> str:
        """Modo 1: simplifica el texto en bullets con frases cortas."""
        prompt = f"""Rewrite the text in simple words and short sentences.
Write each idea as a bullet point starting with "-".

Text: {texto}

Simple version:
-"""
        return self._generar_respuesta(prompt)

    def adaptar_examen(self, texto: str) -> list:
        """Modo 2: coge preguntas de examen y las reescribe más simples (pensado para alumnado con TEA)."""
        preguntas = [q.strip() for q in texto.split("\n") if q.strip()]
        resultados = []

        for q in preguntas:
            prompt = f"""
Rewrite this exam question in simple English for ASD students.

Rules:
- Keep the meaning
- Use very simple words
- Use ONE short sentence only

Output format:
Simplified question: ...

Question:
{q}

Simplified question:
"""
            out = self._generar_respuesta(prompt)
            resultados.append(out)

        return resultados

    def lectura_facil(self, texto: str) -> str:
        """Modo 3: reescribe el texto siguiendo las pautas de "lectura fácil"."""
        prompt = f"""Rewrite the text in Easy-to-Read style.
Rules: short sentences, simple words, one idea per sentence, no metaphors, no abbreviations.

Text: {texto}

Easy-to-Read version:
-"""
        return self._generar_respuesta(prompt)

### Clase: GlosarioGenerator
Detecta palabras difíciles del texto y genera un glosario con explicaciones sencillas usando el mismo modelo de lenguaje.

In [5]:

class GlosarioGenerator:
    """Detecta palabras poco frecuentes y genera un glosario de definiciones simples."""

    def __init__(self, simplifier: TextSimplifier, umbral_frecuencia: float = 3.3):
        self.simplifier = simplifier
        self.umbral_frecuencia = umbral_frecuencia
        self.nlp = spacy.load("en_core_web_sm")

    def detectar_palabras_dificiles(self, texto: str) -> list:
        """Recorre el texto y busca sustantivos poco frecuentes (candidatos a glosario)."""
        doc = self.nlp(texto)

        dificiles = []
        vistas = set()

        for token in doc:
            palabra = token.text.lower()

            # el problema esta en los sustantivos por lo q filtro
            if token.pos_ not in ("NOUN", "PROPN"):
                continue

            if palabra in vistas or len(palabra) < 4:
                continue

            if zipf_frequency(palabra, "en") < self.umbral_frecuencia:
                dificiles.append(palabra)
                vistas.add(palabra)

        return dificiles

    @staticmethod
    def es_definicion_valida(palabra: str, definicion: str) -> bool:
        """Comprueba que la definición generada por el modelo tiene sentido (no vacía ni circular)."""
        definicion_lower = definicion.lower().strip()
        palabra_lower = palabra.lower()

        # Vacía o sospechosamente corta
        if len(definicion_lower.split()) < 3:
            return False

        # Circular: repite la palabra como parte central de la definición
        if definicion_lower == palabra_lower:
            return False
        if definicion_lower.startswith(palabra_lower):
            return False

        return True

    def generar_glosario(self, texto: str) -> dict:
        """Genera el diccionario {palabra: definición} para las palabras difíciles del texto."""
        palabras = self.detectar_palabras_dificiles(texto)
        glosario = {}

        for palabra in palabras:
            prompt = f"""Explain the word in very simple words, like talking to a child.
        Do not repeat the word in your answer. Use one short sentence.

        Word: sun
        Explanation: It is the bright light in the sky during the day.

        Word: bicycle
        Explanation: It is a vehicle with two wheels that you ride by pedaling.

        Word: {palabra}
        Explanation:"""

            explicacion = self.simplifier._generar_respuesta(prompt).strip()

            if not self.es_definicion_valida(palabra, explicacion):
                explicacion = "(Simple explanation not available for this word)"

            glosario[palabra] = explicacion

        return glosario

    @staticmethod
    def formatear(glosario: dict) -> str:
        """Convierte el diccionario del glosario en un texto legible para mostrar en pantalla."""
        glosario_filtrado = {
            p: d for p, d in glosario.items()
            if d != "(Simple explanation not available for this word)"
        }

        if not glosario_filtrado:
            return "No difficult words found."

        lineas = ["Glossary\n"]
        for palabra, definicion in glosario_filtrado.items():
            lineas.append(f"• {palabra.capitalize()}: {definicion}")

        return "\n".join(lineas)

###Clase: TextToSpeech
Convierte el texto adaptado en un archivo de audio (TTS)

In [6]:
class TextToSpeech:
    """Genera archivos de audio a partir de texto usando kokoro."""

    def __init__(self, lang_code: str = "b", voice: str = "af_heart", speed: float = 0.9):
        # lang_code='b' -> inglés británico
        self.pipeline = KPipeline(lang_code=lang_code)
        self.voice = voice
        self.speed = speed

    def generar_audio(self, texto: str) -> str:
        nombre_archivo = f"audioPAEdu_{datetime.now().strftime('%Y%m%d_%H%M%S')}.mp3"
        ruta = f"/content/{nombre_archivo}"

        generator = self.pipeline(
            texto,
            voice=self.voice,
            speed=self.speed
        )

        for _, _, audio in generator:
            sf.write(ruta, audio, 24000)
            break

        return ruta

### Clase: PDFExporter
Genera el PDF final con el texto original y el texto adaptado, usando el estilo visual de la app (colores, logo, pie de página).

In [7]:
class PDFExporter:
    """Genera el PDF de salida con el texto original y el texto adaptado."""

    COLOR_PRIMARIO = colors.HexColor("#2E5EAA")
    COLOR_SECUNDARIO = colors.HexColor("#6C757D")

    def __init__(self, logo_path: str = "/content/drive/MyDrive/4to/TFG/logo_app.png"):
        self.logo_path = logo_path
        self.estilos = self._crear_estilos()

    def _crear_estilos(self) -> dict:
        """Define los estilos de texto que se usan en el PDF (título, subtítulo, cuerpo, etc.)."""
        base = getSampleStyleSheet()
        return {
            "titulo": ParagraphStyle(
                "titulo", parent=base["Title"], fontName="Helvetica-Bold",
                fontSize=20, leading=24, textColor=self.COLOR_PRIMARIO,
                alignment=TA_CENTER, spaceAfter=6
            ),
            "subtitulo": ParagraphStyle(
                "subtitulo", parent=base["Normal"], fontName="Helvetica",
                fontSize=12, leading=16, textColor=self.COLOR_SECUNDARIO,
                alignment=TA_CENTER, spaceAfter=4
            ),
            "fecha": ParagraphStyle(
                "fecha", parent=base["Normal"], fontName="Helvetica-Oblique",
                fontSize=9, textColor=self.COLOR_SECUNDARIO,
                alignment=TA_CENTER, spaceAfter=14
            ),
            "seccion": ParagraphStyle(
                "seccion", parent=base["Heading2"], fontName="Helvetica-Bold",
                fontSize=13, leading=16, textColor=colors.white,
                alignment=TA_CENTER
            ),
            "cuerpo": ParagraphStyle(
                "cuerpo", parent=base["Normal"], fontName="Helvetica",
                fontSize=10.5, leading=15, alignment=TA_JUSTIFY, spaceAfter=14
            ),
        }

    def _etiqueta_seccion(self, texto: str) -> Table:
        """Franja de color con el título de la sección dentro (como una 'card header')."""
        tabla = Table([[Paragraph(texto, self.estilos["seccion"])]], colWidths=[16.2 * cm])
        tabla.setStyle(TableStyle([
            ("BACKGROUND", (0, 0), (-1, -1), self.COLOR_PRIMARIO),
            ("TOPPADDING", (0, 0), (-1, -1), 7),
            ("BOTTOMPADDING", (0, 0), (-1, -1), 7),
            ("LEFTPADDING", (0, 0), (-1, -1), 10),
        ]))
        return tabla

    def _pie_pagina(self, canvas_obj, doc):
        """Pie de página que se repite en todas las hojas del PDF (línea, texto y número de página)."""
        canvas_obj.saveState()
        canvas_obj.setStrokeColor(self.COLOR_SECUNDARIO)
        canvas_obj.line(2.2 * cm, 1.6 * cm, doc.pagesize[0] - 2.2 * cm, 1.6 * cm)

        canvas_obj.setFont("Helvetica-Oblique", 8)
        canvas_obj.setFillColor(self.COLOR_SECUNDARIO)
        canvas_obj.drawString(2.2 * cm, 1.2 * cm,
                               "Generado por PAEdu · Plataforma de Accesibilidad Educativa")
        canvas_obj.drawRightString(doc.pagesize[0] - 2.2 * cm, 1.2 * cm,
                                    f"Página {doc.page}")
        canvas_obj.restoreState()

    def exportar_pdf(self, texto_original: str, texto_resultado: str, modo: str) -> str:
        """Genera el PDF final con el texto original y el texto adaptado."""
        nombre_archivo = f"PAEdu_{datetime.now().strftime('%Y%m%d_%H%M%S')}.pdf"
        ruta = f"/content/{nombre_archivo}"

        doc = SimpleDocTemplate(
            ruta, pagesize=letter,
            topMargin=2.3 * cm, bottomMargin=2.3 * cm,
            leftMargin=2.2 * cm, rightMargin=2.2 * cm,
        )

        elementos = []

        # ----- Logo -----
        try:
            elementos.append(Image(self.logo_path, width=3 * cm, height=3 * cm, hAlign="CENTER"))
            elementos.append(Spacer(1, 8))
        except Exception:
            pass  # si no encuentra el logo, sigue sin romper el documento

        # ----- Cabecera -----
        elementos.append(Paragraph("ASISTENTE EDUCATIVO INCLUSIVO", self.estilos["titulo"]))
        elementos.append(Paragraph(f"Modo de adaptación: <b>{escape(modo)}</b>", self.estilos["subtitulo"]))

        elementos.append(HRFlowable(width="100%", thickness=1, color=self.COLOR_SECUNDARIO, spaceAfter=18))

        # ----- Texto original -----
        elementos.append(self._etiqueta_seccion("TEXTO ORIGINAL"))
        elementos.append(Spacer(1, 10))
        elementos.append(Paragraph(escape(texto_original).replace("\n", "<br/>"), self.estilos["cuerpo"]))
        elementos.append(Spacer(1, 16))

        # ----- Texto adaptado -----
        elementos.append(self._etiqueta_seccion("TEXTO ADAPTADO"))
        elementos.append(Spacer(1, 10))
        elementos.append(Paragraph(escape(texto_resultado).replace("\n", "<br/>"), self.estilos["cuerpo"]))

        doc.build(elementos, onFirstPage=self._pie_pagina, onLaterPages=self._pie_pagina)

        return ruta

### Clase: PAEduApp
Junta todos los componentes anteriores y monta la interfaz de Gradio. Es el "controlador" de toda la aplicación.

In [8]:
class PAEduApp:
    """Clase principal que conecta el modelo, el glosario, el TTS, el exportador y la interfaz."""

    def __init__(self):
        # Monto Drive para poder acceder al logo de la app
        drive.mount('/content/drive', force_remount=True)
        logo = "/content/drive/MyDrive/4to/TFG/logo_app.png"
        print(os.path.exists(logo))  # compruebo que la ruta del logo es correcta

        # Instancio todos los componentes una sola vez (el modelo tarda en cargar)
        self.reader = PDFReader()
        self.simplifier = TextSimplifier()
        self.glosario_generator = GlosarioGenerator(self.simplifier)
        self.tts = TextToSpeech()
        self.exporter = PDFExporter(logo_path=logo)

        self.demo = self._construir_interfaz()

    def procesar(self, texto: str, modo: str):
        """Función 'router': según el modo elegido en la interfaz, llama a un método u otro."""
        if not texto or texto.strip() == "":
            return "Introduce texto o sube un PDF."

        if modo == "Simplificar":
            return self.simplifier.simplificar(texto)

        elif modo == "Adaptar pregunta":
            return self.simplifier.adaptar_examen(texto)

        elif modo == "Lectura fácil":
            return self.simplifier.lectura_facil(texto)

        elif modo == "Glosario":
            glosario = self.glosario_generator.generar_glosario(texto)
            return self.glosario_generator.formatear(glosario)

        return "Modo no válido."

    def run_pipeline(self, texto: str, modo: str):
        """Pipeline completo: procesa el texto según el modo y genera también el audio."""
        if not texto:
            return "Por favor, introduce texto o sube un PDF.", None
        resultado = self.procesar(texto, modo)
        audio_path = self.tts.generar_audio(resultado)
        return resultado, audio_path

    def exportar(self, entrada_texto: str, salida_texto: str, modo: str):
        """Genera el PDF descargable cada vez que cambia el texto de salida."""
        if not salida_texto:
            return None
        return self.exporter.exportar_pdf(entrada_texto, salida_texto, modo)

    def _construir_interfaz(self) -> gr.Blocks:
        """Monta toda la interfaz de la aplicación con Gradio Blocks."""

        theme = gr.themes.Soft(
            primary_hue="pink",
            secondary_hue="rose",
            neutral_hue="slate",
            radius_size="md",
        ).set(
            body_background_fill="#faf7f8",
            block_background_fill="white",
            block_border_width="0.5px",
            block_border_color="#e9c9d1",
        )

        custom_css = """
        .header-container {
            text-align: center;
            padding: 25px 0 30px;
            margin-bottom: 15px;
        }
        .main-title {
            font-size: 25px;
            font-weight: 700;
            color: #ec4899;
        }
        .sub-title {
            margin-top: 8px;
            font-size: 15px;
            color: #6b7280;
        }
        .action-btn {
            font-weight: 600 !important;
            margin-top: 8px !important;
        }

        /* BLOQUES TEXTO */
        textarea {
            height: 420px !important;
            overflow-y: auto !important;
        }

        /* para evitar micro scroll horizontal */
        .gradio-container {
            overflow-x: hidden;
        }
        """

        with gr.Blocks(theme=theme, css=custom_css) as demo:

            # Header
            gr.Markdown("""
            <div class="header-container">
                <div class="main-title">Plataforma de Accesibilidad Educativa</div>
                <div class="sub-title">Asistente de IA para la transformación y adaptación de documentos</div>
            </div>
            """)

            with gr.Row():

                # panel de control/columna izq
                with gr.Column(scale=1, min_width=300):

                    # === SECCIÓN 1: Carga de PDF ===
                    with gr.Group():
                        gr.Markdown("### Configuración")
                        pdf_input = gr.File(
                            label="",
                            file_types=[".pdf"],
                            file_count="single",
                            height=120
                        )

                    # elección de modo y botón para enviar
                    modo = gr.Dropdown(
                        choices=["Simplificar", "Adaptar pregunta", "Lectura fácil", "Glosario"],
                        value="Simplificar",
                        label="Modo de Adaptación",
                        interactive=True
                    )

                    boton = gr.Button(
                        "Transformar",
                        variant="primary",
                        elem_classes="action-btn",
                        size="lg"
                    )

                # columna dcha/visualizacion de contenido
                with gr.Column(scale=2):
                    with gr.Tabs():

                        with gr.TabItem("📝 Textos"):
                            with gr.Row():
                                # texto original
                                with gr.Column():
                                    gr.Markdown("#### Texto Original")
                                    entrada = gr.Textbox(
                                        label="",
                                        placeholder="El texto extraído del PDF aparecerá aquí...",
                                        lines=1,
                                        max_lines=20,
                                        elem_classes="textarea",
                                        show_label=False
                                    )

                                # texto adaptado
                                with gr.Column():
                                    gr.Markdown("#### Texto Adaptado")
                                    salida = gr.Textbox(
                                        label="",
                                        placeholder="El resultado se mostrará aquí...",
                                        lines=1,
                                        max_lines=20,
                                        interactive=False,
                                        elem_classes="textarea",
                                        show_label=False
                                    )

                        # pestaña 2. audio
                        with gr.TabItem("🎧 Audio"):
                            gr.Markdown("### 🔊 Reproductor de Voz")
                            gr.Markdown("Una vez finalice la transformación, el audio estará disponible.")
                            audio = gr.Audio(
                                label="Audio del Texto Adaptado",
                                type="filepath",
                                #show_download_button=True
                            )

                        # pestaña 3. doc generado
                        with gr.TabItem("📥 Documento Generado"):
                            gr.Markdown("### 📄 Descargar PDF Adaptado")
                            gr.Markdown("Tu archivo estará listo para descargar aquí abajo.")
                            descargar = gr.File(
                                label="",
                                interactive=False,
                                height=80
                            )

            # eventos y logica

            # cuando se sube un PDF, se extrae el texto automáticamente
            pdf_input.change(
                fn=self.reader.leer_pdf,
                inputs=pdf_input,
                outputs=entrada
            )

            # al pulsar el botón se ejecuta el pipeline completo (adaptar texto + generar audio)
            boton.click(
                fn=self.run_pipeline,
                inputs=[entrada, modo],
                outputs=[salida, audio]
            )

            # cada vez que cambia el texto de salida, se genera automáticamente el PDF descargable
            salida.change(
                fn=self.exportar,
                inputs=[entrada, salida, modo],
                outputs=descargar
            )

        return demo

    def launch(self, **kwargs):
        """Lanza la interfaz de Gradio."""
        self.demo.launch(**kwargs)

### Punto de entrada de la plataforma

In [ ]:
app = PAEduApp()
app.launch(share=True, debug=True)

Mounted at /content/drive
True


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

/tmp/ipykernel_8064/2537426304.py:101: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=theme, css=custom_css) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://3bb4f917c7fa94ddbb.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


voices/af_heart.pt:   0%|          | 0.00/523k [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/di